# **Space X Falcon 9 First Stage Landing Prediction**
## Machine Learning Prediction 
### Works in Google Colab & Regular Jupyter Notebook

## STEP 1 - Install Libraries (only needed in Colab)

In [ ]:
# Run this cell first - installs all required libraries
!pip install pandas numpy seaborn scikit-learn matplotlib --quiet
print('All libraries installed!')

## STEP 2 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

print('All libraries imported!')

## STEP 3 - Confusion Matrix Function

In [ ]:
def plot_confusion_matrix(y, y_predict):
    "this function plots the confusion matrix"
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, ax=ax, fmt='g')
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title('Confusion Matrix')
    ax.xaxis.set_ticklabels(['did not land', 'land'])
    ax.yaxis.set_ticklabels(['did not land', 'landed'])
    plt.show()

print('Confusion matrix function ready!')

## STEP 4 - Load Data (Fixed - No fetch needed!)

In [ ]:
# Load dataset_part_2 (contains Class/target column)
URL1 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
data = pd.read_csv(URL1)
print('data loaded! Shape:', data.shape)
data.head()

In [ ]:
# Load dataset_part_3 (contains features X)
URL2 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv'
X = pd.read_csv(URL2)
print('X loaded! Shape:', X.shape)
X.head()

## TASK 1 - Create Y array from Class column

In [ ]:
# Create numpy array from Class column
Y = data['Class'].to_numpy()

print('Y array:', Y)
print('Type:', type(Y))
print('Shape:', Y.shape)
print('Unique values:', np.unique(Y))
print('\n0 = Failed landing')
print('1 = Successful landing')

## TASK 2 - Standardize the data X

In [ ]:
# Standardize features using StandardScaler
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)

print(' X standardized!')
print('X shape:', X.shape)
print('Mean (should be ~0):', X.mean().round(2))
print('Std  (should be ~1):', X.std().round(2))

## TASK 3 - Split data into Training and Test sets

In [ ]:
# Split data - 80% train, 20% test
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=2
)

print(' Data split complete!')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'Y_train shape: {Y_train.shape}')
print(f'Y_test shape:  {Y_test.shape}')

In [ ]:
# Check test samples
print('Number of test samples:', Y_test.shape)

## TASK 4 - Logistic Regression with GridSearchCV

In [ ]:
parameters = {'C': [0.01, 0.1, 1],
              'penalty': ['l2'],
              'solver': ['lbfgs']}

lr = LogisticRegression(max_iter=1000)
logreg_cv = GridSearchCV(lr, parameters, cv=10)
logreg_cv.fit(X_train, Y_train)

print('Logistic Regression trained!')
print('Tuned hyperparameters (best parameters):', logreg_cv.best_params_)
print('Accuracy (train/validation):', logreg_cv.best_score_)

## TASK 5 - Logistic Regression Test Accuracy & Confusion Matrix

In [ ]:
# Calculate accuracy on test data
logreg_score = logreg_cv.score(X_test, Y_test)
print('Logistic Regression Test Accuracy:', logreg_score)
print(f'Accuracy: {logreg_score*100:.1f}%')

In [ ]:
# Plot confusion matrix
yhat = logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## TASK 6 - SVM with GridSearchCV

In [ ]:
parameters = {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
              'C': np.logspace(-3, 3, 5),
              'gamma': np.logspace(-3, 3, 5)}

svm = SVC()
svm_cv = GridSearchCV(svm, parameters, cv=10)
svm_cv.fit(X_train, Y_train)

print('SVM trained!')
print('Tuned hyperparameters (best parameters):', svm_cv.best_params_)
print('Accuracy (train/validation):', svm_cv.best_score_)

## TASK 7 - SVM Test Accuracy & Confusion Matrix

In [ ]:
# Calculate accuracy on test data
svm_score = svm_cv.score(X_test, Y_test)
print('SVM Test Accuracy:', svm_score)
print(f'Accuracy: {svm_score*100:.1f}%')

In [ ]:
# Plot confusion matrix
yhat = svm_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## TASK 8 - Decision Tree with GridSearchCV

In [ ]:
parameters = {'criterion': ['gini', 'entropy'],
              'splitter': ['best', 'random'],
              'max_depth': [2*n for n in range(1, 10)],
              'max_features': ['auto', 'sqrt'],
              'min_samples_leaf': [1, 2, 4],
              'min_samples_split': [2, 5, 10]}

tree = DecisionTreeClassifier()
tree_cv = GridSearchCV(tree, parameters, cv=10)
tree_cv.fit(X_train, Y_train)

print('Decision Tree trained!')
print('Tuned hyperparameters (best parameters):', tree_cv.best_params_)
print('Accuracy (train/validation):', tree_cv.best_score_)

## TASK 9 - Decision Tree Test Accuracy & Confusion Matrix

In [ ]:
# Calculate accuracy on test data
tree_score = tree_cv.score(X_test, Y_test)
print('Decision Tree Test Accuracy:', tree_score)
print(f'Accuracy: {tree_score*100:.1f}%')

In [ ]:
# Plot confusion matrix
yhat = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## TASK 10 - KNN with GridSearchCV

In [ ]:
parameters = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'p': [1, 2]}

KNN = KNeighborsClassifier()
knn_cv = GridSearchCV(KNN, parameters, cv=10)
knn_cv.fit(X_train, Y_train)

print(' KNN trained!')
print('Tuned hyperparameters (best parameters):', knn_cv.best_params_)
print('Accuracy (train/validation):', knn_cv.best_score_)

## TASK 11 - KNN Test Accuracy & Confusion Matrix

In [ ]:
# Calculate accuracy on test data
knn_score = knn_cv.score(X_test, Y_test)
print('KNN Test Accuracy:', knn_score)
print(f'Accuracy: {knn_score*100:.1f}%')

In [ ]:
# Plot confusion matrix
yhat = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## TASK 12 - Find Best Model

In [ ]:
# Compare all models
results = {
    'Logistic Regression': logreg_cv.score(X_test, Y_test),
    'SVM':                 svm_cv.score(X_test, Y_test),
    'Decision Tree':       tree_cv.score(X_test, Y_test),
    'KNN':                 knn_cv.score(X_test, Y_test)
}


print('      FINAL MODEL COMPARISON')
print('='*50)
for name, score in results.items():
    print(f'  {name:<25} {score:.4f} ({score*100:.1f}%)')

best_model = max(results, key=results.get)

print(f'BEST MODEL : {best_model}')
print(f'ACCURACY   : {results[best_model]*100:.1f}%')


In [ ]:
# Plot bar chart comparison
import matplotlib.pyplot as plt

names = list(results.keys())
scores = list(results.values())
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

plt.figure(figsize=(10, 6))
bars = plt.bar(names, scores, color=colors, alpha=0.8, edgecolor='black')

for bar, score in zip(bars, scores):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.005,
        f'{score*100:.1f}%',
        ha='center', va='bottom',
        fontsize=12, fontweight='bold'
    )

plt.title('SpaceX Landing Prediction\nModel Accuracy Comparison', 
          fontsize=14, fontweight='bold')
plt.ylabel('Accuracy', fontsize=12)
plt.ylim(0, 1.1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()
print('Chart saved!')


















